# Exercício sobre transfer learning e ajuste fino de um modelo

Nesse exercício, você irá criar um modelo que detecte se uma foto foi tirada em uma área urbana ou rural. Você poderia construir um novo modelo do zero para esse propósito específico, mas para obter bons resultados, você precisaria de milhares de fotos com rótulos de quais são urbanas e quais são rurais.

Você usará *transfer learning* para treinar um modelo pré-treinado com muito menos dados.

ImageNet é um grande conjunto de dados de imagens, composto por mais de 14 milhões de imagens de milhares de categorias. Keras disponibiliza vários modelos que foram pré-treinados neste conjunto de dados aqui. Um dos modelos é o ResNet.

Você irá adaptar o modelo ResNet50 pré-treinado na base de dados *imagenet* para uma nova tarefa de prever se uma imagem é de uma área rural ou urbana.

Para mais informações sobre o modelo, acesse o link para a documentação da classe: [ResNet50](https://keras.io/api/applications/resnet/#resnet50-function)

**OBS**.: Use o exemplo [transfer_learning.ipynb](https://colab.research.google.com/github/zz4fap/tp557-iot-ml/blob/master/examples/transfer_learning.ipynb) como base para resolver este exercício.

## Importando as bibliotecas

Execute a célula de código abaixo.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf

## Pré-processamento dos dados

### Baixe a base de dados

**OBS**.: Como o conjunto de dados é bastante reduzido, usaremos apenas os conjuntos de treinamento e validação. O conjunto de treinamento possui 72 imagens e o de validação apenas 20.

Execute a célula de código abaixo.

In [ ]:
_URL = 'https://www.dropbox.com/scl/fi/xfrxie8y8xhie2t9ziz27/rural_urban_dataset.zip?rlkey=c1x7i6pk4q86imm4mcstqn5yk&dl=1'

path_to_zip = tf.keras.utils.get_file('rural_urban_dataset.zip', origin=_URL, extract=True)

PATH = os.path.join(os.path.dirname(path_to_zip), 'rural_and_urban_photos')
print(PATH)

train_dir = os.path.join(PATH, 'train')
validation_dir = os.path.join(PATH, 'val')

# Mini-batch size.
BATCH_SIZE = 10

# Image size.
IMG_SIZE = (224, 224)

train_dataset = tf.keras.utils.image_dataset_from_directory(train_dir,
                                                            shuffle=True,
                                                            batch_size=BATCH_SIZE,
                                                            image_size=IMG_SIZE)

6450132/6450132 [==============================] - 2s 0us/step


Execute a célula de código abaixo.

In [ ]:
validation_dataset = tf.keras.utils.image_dataset_from_directory(validation_dir,
                                                                 shuffle=True,
                                                                 batch_size=BATCH_SIZE,
                                                                 image_size=IMG_SIZE)

Exibindo as primeiras nove imagens e rótulos do conjunto de treinamento.

Execute a célula de código abaixo.

In [ ]:
class_names = train_dataset.class_names

plt.figure(figsize=(10, 10))
for images, labels in train_dataset.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]]+': '+str(int(labels[i])))
    plt.axis("off")

### Configurando o conjunto de dados para desempenho

Use a pré-busca (i.e., *prefetch*) em buffer para carregar imagens do disco sem bloquear a E/S.

Execute a célula de código abaixo.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)

### Use data augmentation

Aplique transformações aleatórias às imagens de treinamento.

In [ ]:
# Digite o código aqui.

Aplique essas camadas de transformações repetidamente em uma mesma imagem e exiba o resultado.

In [ ]:
# Digite o código aqui.

### Redimensione os valores de pixel

Use a função `tf.keras.applications.resnet.preprocess_input` para converter as imagens de entrada de RGB para BGR e, em seguida, centralizar cada canal de cores em relação ao conjunto de dados ImageNet, sem dimensionamento.

Para mais informações sobre o modelo, acesse o link para a documentação da classe: [ResNet50](https://keras.io/api/applications/resnet/#resnet50-function)

In [ ]:
# Digite o código aqui.

## Crie o modelo base a partir das convnets pré-treinadas

Instancie o modelo ResNet50 com `include_top=False` e `weights='imagenet'`. Lembre-se que o `input_shape` deve ter 3 dimensões.

In [ ]:
# Digite o código aqui.

## Extração de características

Nesta etapa, você deve congelar a base convolucional do modelo criado na etapa anterior e usá-la como extrator de características. Além disso, você deve adicionar um classificador sobre ele e treiná-lo.

### Congele a base convolucional


In [ ]:
# Digite o código aqui.

### Imprima o resumo do modelo e compare a quantidade de parâmetros treináveis e não treináveis.

In [ ]:
# Digite o código aqui.

### Adicione uma camada de classificação

Gere predições a partir do bloco de características tomando a média das localizações espaciais usando uma camada `tf.keras.layers.GlobalAveragePooling2D` para converter as características em um vetor.

In [ ]:
# Digite o código aqui.

Na sequência, crie uma camada `tf.keras.layers.Dense` para converter essas características em uma única predição por imagem. Não precisamos de uma função de ativação aqui porque esta predição será tratada como um `logit`, ou seja, um valor bruto de predição. Os números positivos predizem a classe 1, os números negativos predizem a classe 0.

In [ ]:
# Digite o código aqui.

Crie o modelo encadeando as camadas de aumento de dados, pré-processamento, modelo base (ResNet50), `global_average_layer` e classificador. Adicione uma camada de `dropout` entre a camada `global_average_layer` e a camada de classificação.

Não se esqueça de fazer `training=False` no modelo base (ResNet50), pois o modelo contém camadas de `BatchNormalization`.

In [ ]:
# Digite o código aqui.

### Imprima o resumo do modelo e compare a quantidade de parâmetros treináveis e não treináveis.

In [ ]:
# Digite o código aqui.

### Compile o modelo

Compile o modelo antes de treiná-lo. Como existem duas classes, use a perda `tf.keras.losses.BinaryCrossentropy` com `from_logits=True`, pois o modelo fornece uma saída linear (i.e., última camada não tem função de ativação).

**OBS**.: Use um passo de aprendizagem de `0.0001`

In [ ]:
# Digite o código aqui.

### Treine o modelo

Verifique a acurácia do modelo sem nenhum treinamento no conjunto de validação.

In [ ]:
# Digite o código aqui.

Treine o modelo por 25 épocas. Você deve ver uma acurácia de aproximadamente 90% no conjunto de validação.

In [ ]:
# Digite o código aqui.

### Plote as curvas de aprendizado

Analise as curvas de aprendizado da acurácia/perda de treinamento e validação ao usar o modelo base ResNet50 como um extrator de características.

In [ ]:
# Digite o código aqui.

## Faça o ajuste fino do modelo


### Descongele as camadas superiores do modelo

Tudo o que você precisa fazer é descongelar o modelo base e definir as camadas inferiores como não treináveis. Em seguida, você deve recompilar o modelo (necessário para que essas alterações tenham efeito) e retomar o treinamento.

### Descongele o modelo base.

In [ ]:
# Digite o código aqui.

### Congele apenas as 100 primeiras camadas do modelo base.

In [ ]:
# Digite o código aqui.

### Compile o modelo

Como você está treinando um modelo muito maior e deseja reajustar os pesos pré-treinados, é importante utilizar uma taxa de aprendizado menor nesta fase. Caso contrário, o modelo poderá se ajustar muito rapidamente.

**OBS**.: Use um passo de aprendizagem 10 vezes menor do que o usado anteriormente.

In [ ]:
# Digite o código aqui.

### Imprima o resumo do modelo e compare a quantidade de parâmetros treináveis e não treináveis.

In [ ]:
# Digite o código aqui.

### Continue o treinamento do modelo

Esta etapa deve melhorar a acurácia do modelo em alguns pontos percentuais.

**OBS**.: Faça o ajuste fine do modelo por 15 épocas.

In [ ]:
# Digite o código aqui.

### Plote as curvas de aprendizado

Analise as curvas de aprendizado da acurácia/perda de treinamento e validação ao usar o modelo base ResNet50 como um extrator de características.

Após o ajuste fino, o modelo deve atingir mais de 98% de acurácia no conjunto de validação.

In [ ]:
# Digite o código aqui.

### Avalie e faça predições com o modelo

Verifique o desempenho do modelo nos dados do conjunto de validação usando o método `evaluate`.

In [ ]:
# Digite o código aqui.

### Exiba algumas imagens do conjunto de validação juntamente com a classe que foi atribuída a ela.

In [ ]:
# Digite o código aqui.

### Após análisar os resultados gerados pelo novo modelo, o que você pode concluir?

**Resposta**

Escreva sua reposta aqui.